# Notebook 01: Procesos, Hilos y el GIL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/01_procesos_hilos.ipynb)

**Módulo 16 — Clase 1**

Este notebook acompaña los archivos `01_procesos_y_hilos.md` y `02_secuencial.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [1]:
import os
import sys
import time
import threading
import multiprocessing

print(f'Python {sys.version}')
print(f'CPU cores: {os.cpu_count()}')

Python 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:52:34) [Clang 18.1.8 ]
CPU cores: 8


## Los escenarios del chatbot

A lo largo del módulo trabajaremos con dos escenarios de nuestro chatbot:

- **Escenario A** — LLM como API remota: `recv(1ms exec) → BD(50ms wait) → LLM API(1500ms wait) → send(5ms wait)` → I/O-bound → M4
- **Escenario B** — LLM local: `recv(1ms exec) → BD(50ms wait) → inferencia(2000ms exec) → send(5ms wait)` → CPU-bound → M5b

Este notebook establece los conceptos fundamentales que hacen posible esa distinción: procesos, hilos y el GIL.


## Sección 1: Inspeccionando un proceso

Cada programa Python que ejecutas ES un proceso con su propio PID y espacio de memoria.

In [2]:
print(f'Mi PID (Process ID): {os.getpid()}')
print(f'PID de mi proceso padre: {os.getppid()}')

# Intentar instalar psutil si no está disponible
try:
    import psutil
    proc = psutil.Process(os.getpid())
    mem = proc.memory_info()
    print(f'Memoria RSS: {mem.rss / 1024**2:.1f} MB')
    print(f'Memoria VMS: {mem.vms / 1024**2:.1f} MB')
except ImportError:
    print('psutil no instalado — pip install psutil para más info')

Mi PID (Process ID): 42496
PID de mi proceso padre: 41930
Memoria RSS: 37.2 MB
Memoria VMS: 401467.2 MB


## Sección 2: Inspeccionando hilos

El hilo principal es θ_main. Podemos crear hilos adicionales — todos comparten el heap del proceso.

In [3]:
print(f'Hilo actual: {threading.current_thread().name}')
print(f'ID del hilo: {threading.current_thread().ident}')
print(f'Hilos activos: {threading.active_count()}')

# Crear un hilo que reporte su identidad
def reportar_identidad(nombre):
    print(f'  [{nombre}] PID={os.getpid()}, Hilo={threading.current_thread().name}, ID={threading.current_thread().ident}')

print('\nLanzando 3 hilos:')
hilos = []
for i in range(3):
    h = threading.Thread(target=reportar_identidad, args=(f'hilo-{i}',))
    hilos.append(h)
    h.start()

for h in hilos:
    h.join()

print('\nObserva: todos los hilos tienen el MISMO PID — son del mismo proceso.')

Hilo actual: MainThread
ID del hilo: 8383389888
Hilos activos: 7

Lanzando 3 hilos:
  [hilo-0] PID=42496, Hilo=Thread-4 (reportar_identidad), ID=6284832768
  [hilo-1] PID=42496, Hilo=Thread-5 (reportar_identidad), ID=6284832768
  [hilo-2] PID=42496, Hilo=Thread-6 (reportar_identidad), ID=6284832768

Observa: todos los hilos tienen el MISMO PID — son del mismo proceso.


## Sección 3: Demostración del GIL — CPU-bound con threading

**Hipótesis:** con 2 hilos y 2 cores, la tarea CPU-bound debería tardar la mitad.  
**Resultado real:** el GIL serializa los hilos → casi no hay speedup (puede ser más lento).

In [4]:
def tarea_cpu_bound(n, nombre=''):
    """Tarea CPU-bound pura: suma de 0 a n. wait(τ) = ∅"""
    resultado = sum(range(n))
    return resultado

N = 30_000_000  # ajusta si es muy lento o muy rápido en tu máquina

# --- Secuencial ---
t0 = time.perf_counter()
tarea_cpu_bound(N)
tarea_cpu_bound(N)
t_secuencial = time.perf_counter() - t0

# --- Threading (2 hilos, misma carga total) ---
t0 = time.perf_counter()
h1 = threading.Thread(target=tarea_cpu_bound, args=(N,))
h2 = threading.Thread(target=tarea_cpu_bound, args=(N,))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

print(f'Secuencial:  {t_secuencial:.2f}s')
print(f'Threading:   {t_threading:.2f}s')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ~2x, real ~1x por el GIL)')
print()
print('Conclusión: para CPU-bound, threading en Python NO ayuda.')
print('El GIL garantiza que solo un hilo ejecuta bytecode a la vez.')

Secuencial:  0.53s
Threading:   0.49s
Speedup:     1.08x  (esperado ~2x, real ~1x por el GIL)

Conclusión: para CPU-bound, threading en Python NO ayuda.
El GIL garantiza que solo un hilo ejecuta bytecode a la vez.


---

## Sección 4: GIL liberado — I/O-bound con threading

Durante `wait(τᵢ)` (I/O), el GIL se libera. Aquí sí hay speedup con threading.

In [5]:
def tarea_io_bound(duracion, nombre=''):
    """Simula I/O-bound: espera bloqueante. wait(τ) ≠ ∅"""
    time.sleep(duracion)  # durante sleep, el GIL se libera

DURACION = 1.0  # cada tarea "espera" 1s de I/O

# --- Secuencial ---
t0 = time.perf_counter()
tarea_io_bound(DURACION)
tarea_io_bound(DURACION)
t_secuencial = time.perf_counter() - t0

# --- Threading ---
t0 = time.perf_counter()
h1 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h2 = threading.Thread(target=tarea_io_bound, args=(DURACION,))
h1.start(); h2.start()
h1.join(); h2.join()
t_threading = time.perf_counter() - t0

print(f'Secuencial:  {t_secuencial:.2f}s  (suma: 2 × {DURACION}s)')
print(f'Threading:   {t_threading:.2f}s  (paralelo en I/O, GIL liberado)')
print(f'Speedup:     {t_secuencial/t_threading:.2f}x  (esperado ~2x)')
print()
print('Conclusión: para I/O-bound, threading SÍ ayuda porque el GIL se libera durante sleep/I/O.')

Secuencial:  2.01s  (suma: 2 × 1.0s)
Threading:   1.01s  (paralelo en I/O, GIL liberado)
Speedup:     2.00x  (esperado ~2x)

Conclusión: para I/O-bound, threading SÍ ayuda porque el GIL se libera durante sleep/I/O.


## Sección 5: Aislamiento de memoria — proceso vs hilo

Los hilos comparten memoria (un cambio es visible para todos).  
Los procesos NO comparten memoria (cada uno tiene su propia copia).

In [6]:
# --- Hilos comparten memoria ---
contador_compartido = [0]  # lista mutable accesible por todos los hilos

def incrementar_hilo(n):
    for _ in range(n):
        contador_compartido[0] += 1  # AMBOS hilos ven y modifican esto

hilos = [threading.Thread(target=incrementar_hilo, args=(1000,)) for _ in range(2)]
for h in hilos: h.start()
for h in hilos: h.join()

print(f'Contador tras 2 hilos × 1000 incrementos: {contador_compartido[0]}')
print('(Nota: puede ser < 2000 debido a race condition — lo veremos en Clase 2)')

print()

# --- Procesos NO comparten memoria ---
def incrementar_proceso(lista_compartida, n):
    # En un proceso hijo, lista_compartida es una COPIA — no afecta al padre
    for _ in range(n):
        lista_compartida[0] += 1
    print(f'  Proceso hijo ({os.getpid()}): su copia = {lista_compartida[0]}')

if __name__ == '__main__':  # necesario para multiprocessing en scripts
    dato = [0]
    p = multiprocessing.Process(target=incrementar_proceso, args=(dato, 1000))
    p.start()
    p.join()
    print(f'Proceso padre ({os.getpid()}): su copia = {dato[0]}')
    print('→ El padre no ve el cambio del hijo: Mem(padre) ∩ Mem(hijo) = ∅')

Contador tras 2 hilos × 1000 incrementos: 2000
(Nota: puede ser < 2000 debido a race condition — lo veremos en Clase 2)

Proceso padre (42496): su copia = 0
→ El padre no ve el cambio del hijo: Mem(padre) ∩ Mem(hijo) = ∅


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/damiangallardoloya/miniforge3/envs/finkargo/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/damiangallardoloya/miniforge3/envs/finkargo/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'incrementar_proceso' on <module '__main__' (built-in)>


---

## Resumen

| Concepto | Clave |
|---|---|
| Proceso | Tiene su propia memoria aislada |
| Hilo | Comparte memoria del proceso |
| GIL | Solo 1 hilo ejecuta bytecode Python a la vez |
| CPU-bound + threading | Sin speedup (GIL nunca se libera) |
| I/O-bound + threading | Con speedup (GIL se libera durante I/O) |